In [ ]:

import json
import pennylane as qml
import pennylane.numpy as np

symbols = ["H", "H", "H"]


def h3_ground_energy(bond_length):
    
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.
    
    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
        state of the H3+ molecule with the given bond length.
    """
    
    # Define the geometry of the H3+ molecule as an equilateral triangle
    # The first H atom is at the origin.
    # The second H atom is along the x-axis at bond_length.
    # The third H atom is positioned to form an equilateral triangle.
    geometry = np.array([
        [0.0, 0.0, 0.0],
        [bond_length, 0.0, 0.0],
        [bond_length / 2, bond_length * np.sqrt(3) / 2, 0.0]
    ], requires_grad=False)

    # H3+ has 3 protons and 2 electrons (3*1 - 1 = 2 electrons)
    # Using STO-3G basis, each H atom contributes 1 spatial orbital.
    # So, 3 spatial orbitals in total, which means 6 spin orbitals (qubits).
    # We have 2 electrons, so the active space will typically involve these 2 electrons.
    
    # Build the molecular Hamiltonian
    # Charge = 1 for H3+
    # active_electrons = 2 (number of electrons in H3+)
    # active_orbitals = 3 (number of spatial orbitals from 3 H atoms in STO-3G)
    H, qubits = qml.qchem.molecular_hamiltonian(
        symbols,
        geometry,
        charge=1,
        basis="STO-3G",
        active_electrons=2,
        active_orbitals=3
    )

    # Determine the Hartree-Fock state
    # n_electrons is the number of active electrons, which is 2 for H3+
    # qubits is the total number of spin orbitals in the active space (2*3 = 6)
    hf_state = qml.qchem.hf_state(n_electrons=2, n_qubits=qubits)

    # Generate all single and double excitations from the Hartree-Fock state
    singles, doubles = qml.qchem.excitations(n_electrons=2, n_qubits=qubits)
    excitations = singles + doubles

    # Define the VQE ansatz circuit
    dev = qml.device("default.qubit", wires=qubits)

    @qml.qnode(dev, interface="autograd")
    def circuit(params, excitations, hf_state, H):
        # Prepare the Hartree-Fock state
        qml.BasisState(hf_state, wires=range(qubits))
        
        # Apply the Unitary Coupled Cluster Singles and Doubles (UCCSD) ansatz
        for i, excitation in enumerate(excitations):
            if len(excitation) == 4: # Double excitation
                qml.DoubleExcitation(params[i], wires=excitation)
            elif len(excitation) == 2: # Single excitation
                qml.SingleExcitation(params[i], wires=excitation)
        
        # Return the expectation value of the Hamiltonian
        return qml.expval(H)

    # Initialize the VQE parameters to zeros
    params = np.array(np.zeros(len(excitations)), requires_grad=True)

    # Set up the optimizer
    opt = qml.GradientDescentOptimizer(stepsize=0.4) # A common step size for VQE

    # Perform the VQE optimization
    num_steps = 100 # Number of optimization steps
    for n in range(num_steps):
        params, energy = opt.step_and_cost(circuit, params, excitations=excitations, hf_state=hf_state, H=H)
        # Optional: Print progress
        # if n % 10 == 0:
        #     print(f"Step {n}, Energy: {energy:.8f}")

    return energy


# These functions are responsible for testing the solution.

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol = 1e-4), "Not the correct ground energy"

